# 03 Demo clips

Built from repo commit `a12f8a8`. Renders the clips chosen by the fixed rule in `configs/experiments.yaml` (`demo`) from cached outputs of notebooks 01 and 02: largest earlier-warning margins of the main model over the detection-only model, plus a false alarm and a miss. Left: detection-only (`det_graph`); right: main model with road context. Box colour = attention weight, i.e. what the model relied on, not what caused the crash.

In [ ]:
import base64, io, zipfile, os, sys, glob, json, subprocess
BUNDLE = '...'
CODE = '/kaggle/tmp/code'; os.makedirs(CODE, exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))).extractall(CODE)
sys.path.insert(0, CODE)
def find(p):
    h = glob.glob(p, recursive=True)
    return h[0] if h else None
FEAT = os.path.dirname(find('/kaggle/input/**/features/ccd_pos_000001.npz'))
CCD_ROOT = os.path.dirname(find('/kaggle/input/**/CCD/Crash-1500.txt'))
DOTA_META = os.path.dirname(find('/kaggle/input/**/dota_official/metadata_train.json'))
print(FEAT, CCD_ROOT, DOTA_META, len(glob.glob(FEAT + '/*.npz')), 'feature files')

In [ ]:
RUNS = os.path.dirname(find('/kaggle/input/**/runs/main'))
SEL = find('/kaggle/input/**/results/tables/demo_selection.json')
ccd_v = find('/kaggle/input/**/CCD/videos'); dota_v = find('/kaggle/input/**/DoTA_STAGNet/videos')
paths = {}
for p in glob.glob(ccd_v + '/Crash-1500/*.mp4'): paths['ccd_pos_' + os.path.basename(p)[:-4]] = p
for p in glob.glob(ccd_v + '/Normal/*.mp4'): paths['ccd_neg_' + os.path.basename(p)[:-4]] = p
for p in glob.glob(dota_v + '/*/positive/*.avi'): paths['dota_pos_' + os.path.basename(p)[:-4]] = p
for p in glob.glob(dota_v + '/*/negative/*.avi'): paths['dota_neg_' + os.path.basename(p)[:-4]] = p
open('/kaggle/tmp/videos.txt', 'w').write('\n'.join(f'{k} {v}' for k, v in paths.items()))
r = subprocess.run([sys.executable, '-m', 'src.demo.render', '--selection', SEL, '--runs', RUNS, '--feat_dir', FEAT,
                    '--video_list', '/kaggle/tmp/videos.txt', '--out', '/kaggle/working/demo'],
                   cwd=CODE, env=dict(os.environ, PYTHONPATH=CODE), capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-2000:])

In [ ]:
from IPython.display import Markdown, Image, Video, display
for f in sorted(glob.glob('/kaggle/working/demo/*.mp4')):
    display(Markdown(f'**{os.path.basename(f)}** ({os.path.getsize(f) / 1e6:.1f} MB)'))
    display(Video(f, embed=True, width=960, html_attributes='controls muted loop'))
for f in sorted(glob.glob('/kaggle/working/demo/*_storyboard.png')):
    display(Markdown(f'**{os.path.basename(f)}**')); display(Image(f, width=960))